## Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 300)

PROJECT_DIR = Path.cwd()
ENCODED_INPUT_DIR = (PROJECT_DIR/ "Cleaned Data"/ "002 Data Clean"/ "cleaned_encoded")
OUTPUT_DIR = (PROJECT_DIR/ "Cleaned Data"/ "003 Column Names update")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COUNTRIES = {"AUS": "Australia","BRA": "Brazil","CAN": "Canada","GBR": "United Kingdom","IND": "India","USA": "United States",}

## 1 Read encoded file

In [2]:
encoded_data = {}

# Read all files
for code, country_name in COUNTRIES.items():
    safe_country_name = country_name.replace(" ", "_")                                      # Read replace " " to "_"
    file_path = (ENCODED_INPUT_DIR/ f"YouGov_{safe_country_name}_cleaned_encoded.csv")      # Path
    encoded_data[code] = pd.read_csv(file_path,encoding="utf-8-sig")                        # use utf-8-sig

    print(f"{country_name}: "f"{encoded_data[code].shape[0]} rows, "f"{encoded_data[code].shape[1]} columns")

Australia: 41745 rows, 67 columns
Brazil: 10118 rows, 130 columns
Canada: 39911 rows, 71 columns
United Kingdom: 32418 rows, 81 columns
India: 15365 rows, 127 columns
United States: 17215 rows, 102 columns


## 2 Readable column name

In [3]:
all_encoded_columns = sorted(set().union(*[set(df.columns) for df in encoded_data.values()]))                   # Sorted columns name


# Old name and new name (1)
readable_rename_dict = {
    "RecordNo": "Record ID",
    "endtime": "Survey Date",
    "age": "Age",
    "household_size": "Household Size",
    "week_number": "Week",
    "within_mandate_period": "Mandate Period",

    "face_mask_behaviour_scale": "Face Mask Wearing Score",
    "face_mask_behaviour_binary": "Face Mask Wearing Binary",
    "non_mask_protective_behaviour_scale": "Non-Mask Protective Behavior Score",
    "non_mask_protective_behaviour_binary": "Non-Mask Protective Behavior Binary",
    "overall_protective_behaviour_scale": "Overall Protective Behavior Score",
    "overall_protective_behaviour_binary": "Overall Protective Behavior Binary",

    "r1_1": "COVID Severity",
    "r1_2": "COVID Infection Risk",
    "r1_6": "Health Motivation",
    "r1_7": "COVID Life Impact",

    "i1_health": "Physical contact count (with family)",
    "i2_health": "Physical contact count (with stranger)",
    "i7a_health": "Outings count",
    "i13_health": "Handwashing Count",

    "gender_Male": "Is a male?"}


# Old name and new name (2)
dummy_name_groups = {
    "CORE_B2_4": {
        "Don't know": "Happiness Change: Unknown",
        "Much less happy now": "Happiness Change: Much Less",
        "Much more happy now": "Happiness Change: Much More",
        "N/A": "Happiness Change: N/A",
        "Somewhat less happy now": "Happiness Change: Somewhat Less",
        "Somewhat more happy now": "Happiness Change: Somewhat More"},

    "WCRV_4": {
        "I am fairly scared that I will contract the Coronavirus (COVID-19)": "COVID Fear: Fairly Scared",
        "I am not at all scared that I will contract the Coronavirus (COVID-19)": "COVID Fear: Not Scared",
        "I am not very scared that I will contract the Coronavirus (COVID-19)": "COVID Fear: Slightly Scared",
        "I am very scared that I will contract the Coronavirus (COVID-19)": "COVID Fear: Very Scared",
        "N/A": "COVID Fear: N/A",
        "Not applicable - I have already contracted Coronavirus (COVID-19)": "COVID Fear: Had COVID"},

    "WCRex1": {
        "N/A": "Gov Response: N/A",
        "Somewhat badly": "Gov Response: Somewhat Bad",
        "Somewhat well": "Gov Response: Somewhat Good",
        "Very badly": "Gov Response: Very Bad",
        "Very well": "Gov Response: Very Good"},

    "WCRex2": {
        "A lot of confidence": "Confidence in the government: High",
        "Don't know": "Confidence in the government: Unknown",
        "N/A": "Confidence in the government: N/A",
        "No confidence at all": "Confidence in the government: None",
        "Not very much confidence": "Confidence in the government: Low"},

    "employment_status": {
        "Full time student": "Employment: Student",
        "Not working": "Employment: Not Working",
        "Other": "Employment: Other",
        "Part time employment": "Employment: Part Time",
        "Retired": "Employment: Retired",
        "Unemployed": "Employment: Unemployed"},

    "d1_comorbidities": {
        "No": "Comorbidity: No",
        "Prefer_not_to_say": "Comorbidity: Prefer Not Say",
        "Yes": "Comorbidity: Yes"},

    "i9_health": {
        "No": "Symptom Isolation: No",
        "Not sure": "Symptom Isolation: Unsure",
        "Yes": "Symptom Isolation: Yes"},

    "i10_health": {
        "Neither easy nor difficult": "Isolation Ease: Neutral",
        "Not sure": "Isolation Ease: Unsure",
        "Somewhat difficult": "Isolation Ease: Somewhat Difficult",
        "Somewhat easy": "Isolation Ease: Somewhat Easy",
        "Very difficult": "Isolation Ease: Very Difficult",
        "Very easy": "Isolation Ease: Very Easy"},

    "i11_health": {
        "Neither willing nor unwilling": "Isolation Willingness: Neutral",
        "Not sure": "Isolation Willingness: Unsure",
        "Somewhat unwilling": "Isolation Willingness: Somewhat Unwilling",
        "Somewhat willing": "Isolation Willingness: Somewhat Willing",
        "Very unwilling": "Isolation Willingness: Very Unwilling",
        "Very willing": "Isolation Willingness: Very Willing"},

    "i3_health": {
        "No, I have not": "COVID Test: Not done",
        "Yes, and I have not received my results from the test yet": "COVID Test: Pending",
        "Yes, and I tested negative": "COVID Test: Negative",
        "Yes, and I tested positive": "COVID Test: Positive"},

    "i4_health": {
        "No, they have not": "Household COVID Test: Not done",
        "Not sure": "Household COVID Test: Unsure",
        "Yes, and they have not received their results from the test yet": "Household COVID Test: Pending",
        "Yes, and they tested negative": "Household COVID Test: Negative",
        "Yes, and they tested positive": "Household COVID Test: Positive"}}


for prefix, value_map in dummy_name_groups.items():
    for old_value, new_name in value_map.items():
        readable_rename_dict[f"{prefix}_{old_value}"] = new_name


# Old name and new name (3.1)
phq_items = {
    "PHQ4_1": "PHQ Low Interest",
    "PHQ4_2": "PHQ Depressed",
    "PHQ4_3": "PHQ Anxiety",
    "PHQ4_4": "PHQ Worry"}

# Old name and new name (3.2)(PHQ4's answer)
phq_values = {
    "N/A": "N/A",
    "Nearly every day": "Daily",
    "Not at all": "Not At All",
    "Prefer not to say": "Prefer Not Say",
    "Several days": "Several Days"}

for item, item_name in phq_items.items():
    for old_value, short_value in phq_values.items():
        readable_rename_dict[f"{item}_{old_value}"] = (f"{item_name}: {short_value}")


# Old name and new name (4)
symptom_items = {
    "i5_health_1": "Dry Cough",
    "i5_health_2": "Fever",
    "i5_health_3": "Loss of Smell",
    "i5_health_4": "Loss of Taste",
    "i5_health_5": "Shortness of Breath",
    "i5_health_99": "No Symptoms"}

for item, item_name in symptom_items.items():
    for value in ["No", "Yes"]:
        readable_rename_dict[f"{item}_{value}"] = (f"{item_name}: {value}")


# Apply readable naming rules
def make_readable_column_name(col):
    if col in readable_rename_dict:
        return readable_rename_dict[col]
    if col.startswith("state_"):
        return "State: " + col.replace("state_", "")
    return col

# Save and display
column_name_update_table = pd.DataFrame({"old_column_name": all_encoded_columns,"new_column_name": [make_readable_column_name(col) for col in all_encoded_columns]})
columns_update_path = OUTPUT_DIR / "Columns update.csv"
column_name_update_table.to_csv(columns_update_path,index=False,encoding="utf-8-sig")

display(column_name_update_table)

,old_column_name,new_column_name
0,CORE_B2_4_Don't know,Happiness Change: Unknown
1,CORE_B2_4_Much less happy now,Happiness Change: Much Less
2,CORE_B2_4_Much more happy now,Happiness Change: Much More
3,CORE_B2_4_N/A,Happiness Change: N/A
4,CORE_B2_4_Somewhat less happy now,Happiness Change: Somewhat Less
5,CORE_B2_4_Somewhat more happy now,Happiness Change: Somewhat More
6,PHQ4_1_N/A,PHQ Low Interest: N/A
7,PHQ4_1_Nearly every day,PHQ Low Interest: Daily
8,PHQ4_1_Not at all,PHQ Low Interest: Not At All
9,PHQ4_1_Prefer not to say,PHQ Low Interest: Prefer Not Say


## 3 Update columns name

In [4]:
rename_dict = dict(zip(column_name_update_table["old_column_name"],column_name_update_table["new_column_name"]))        # Update
readable_encoded_data = {}

for code, country_name in COUNTRIES.items():
    readable_encoded_data[code] = encoded_data[code].rename(columns=rename_dict)                                        # Rename

### 4 Save

In [5]:
for code, country_name in COUNTRIES.items():
    output_path = OUTPUT_DIR / f"{country_name} (readable).csv"                                     # Save all files
    readable_encoded_data[code].to_csv(output_path,index=False,encoding="utf-8-sig")                # utf-8-sig